In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv('../data/processed/TS_history_cleaned.csv', dtype={'app_id': str, 'shop_id': str}, parse_dates=['date'])
game_info = pd.read_csv('../data/raw/TS_game_info.csv', dtype={'app_id': str})

In [3]:
# data['app_id'] = data['app_id'].str.rstrip('.0')

In [4]:
print("=== FEATURE ENGINEERING ===\n")
fe_data = data.copy()
# Ensure date is datetime type
fe_data['date'] = pd.to_datetime(fe_data['date'])

# Sort by app_id, shop_id, and date for time-series calculations
fe_data = fe_data.sort_values(['app_id', 'shop_id', 'date']).reset_index(drop=True)

# Feature 1: Historical price statistics per game/shop
print("1. Creating historical price statistics...")
fe_data['price_min'] = fe_data.groupby(['app_id', 'shop_id'])['price'].transform('min')
fe_data['price_max'] = fe_data.groupby(['app_id', 'shop_id'])['price'].transform('max')
fe_data['price_mean'] = fe_data.groupby(['app_id', 'shop_id'])['price'].transform('mean')
fe_data['price_median'] = fe_data.groupby(['app_id', 'shop_id'])['price'].transform('median')
fe_data['price_std'] = fe_data.groupby(['app_id', 'shop_id'])['price'].transform('std').fillna(0)

# Feature 2: Price quantile for "historically low" determination
fe_data['price_q25'] = fe_data.groupby(['app_id', 'shop_id'])['price'].transform(lambda x: x.quantile(0.25))
fe_data['price_q75'] = fe_data.groupby(['app_id', 'shop_id'])['price'].transform(lambda x: x.quantile(0.75))

# Feature 3: Price position in historical range (0 = min, 1 = max)
fe_data['price_position'] = ((fe_data['price'] - fe_data['price_min']) / 
                          (fe_data['price_max'] - fe_data['price_min'] + 0.01)).clip(0, 1)

# Feature 4: Sale duration
print("2. Calculating sale duration...")
sale_duration = fe_data.groupby(['app_id', 'shop_id']).agg({
    'date': ['min', 'max', 'count']
}).reset_index()
sale_duration.columns = ['app_id', 'shop_id', 'sale_start', 'sale_end', 'sale_records']
sale_duration['sales_duration_days'] = (sale_duration['sale_end'] - sale_duration['sale_start']).dt.days
fe_data = fe_data.merge(sale_duration[['app_id', 'shop_id', 'sales_duration_days']], on=['app_id', 'shop_id'], how='left')

# Feature 5: Last sale date tracking
print("3. Tracking last sale date...")
sale_events = fe_data[fe_data['sales_percentage'] > 0][['app_id', 'shop_id', 'date']].copy()
sale_events = sale_events.rename(columns={'date': 'last_sale_date'})
fe_data = fe_data.merge(sale_events.drop_duplicates(['app_id', 'shop_id'], keep='last'), 
                        on=['app_id', 'shop_id'], how='left')
fe_data['last_sale_date'] = fe_data.groupby(['app_id', 'shop_id'])['last_sale_date'].ffill()
fe_data['days_since_last_sale'] = (fe_data['date'] - fe_data['last_sale_date']).dt.days

# Feature 6: Release date based temporal features
print("4. Adding release date features...")
fe_data = fe_data.merge(game_info[['app_id', 'release_date']], on='app_id', how='left')
fe_data['release_date'] = pd.to_datetime(fe_data['release_date'], errors='coerce')
fe_data['days_since_release'] = (fe_data['date'] - fe_data['release_date']).dt.days

# Feature 7: Seasonal features
print("5. Adding seasonal features...")
fe_data['month'] = fe_data['date'].dt.month
fe_data['quarter'] = fe_data['date'].dt.quarter
fe_data['season'] = fe_data['month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall', 10: 'Fall', 11: 'Fall'
})
season_dummies = pd.get_dummies(fe_data['season'], prefix='season')
fe_data = pd.concat([fe_data, season_dummies], axis=1)
fe_data['is_holiday_season'] = fe_data['month'].isin([11, 12]).astype(int)

# Feature 8: Lagged price features
print("6. Creating lagged price features...")
fe_data['lagged_price_1'] = fe_data.groupby(['app_id', 'shop_id'])['price'].shift(1).fillna(fe_data['price'])
fe_data['lagged_price_7'] = fe_data.groupby(['app_id', 'shop_id'])['price'].shift(7).fillna(fe_data['price'])

# Feature 9: Rolling statistics
print("7. Computing rolling statistics...")
fe_data['price_rolling_mean_7d'] = fe_data.groupby(['app_id', 'shop_id'])['price'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()).fillna(0)
fe_data['price_rolling_std_7d'] = fe_data.groupby(['app_id', 'shop_id'])['price'].transform(
    lambda x: x.rolling(window=7, min_periods=1).std()).fillna(0)
fe_data['price_rolling_mean_30d'] = fe_data.groupby(['app_id', 'shop_id'])['price'].transform(
    lambda x: x.rolling(window=30, min_periods=1).mean()).fillna(0)
fe_data['price_rolling_std_30d'] = fe_data.groupby(['app_id', 'shop_id'])['price'].transform(
    lambda x: x.rolling(window=30, min_periods=1).std()).fillna(0)

# Feature 10: Days since historical low
print("8. Finding days since historical low...")
low_dates = fe_data.loc[
    fe_data['price'].eq(fe_data.groupby(['app_id', 'shop_id'])['price'].transform('min')),
    ['app_id', 'shop_id', 'date']
].groupby(['app_id', 'shop_id'])['date'].min()

fe_data = fe_data.join(low_dates.rename('historical_low_date'), on=['app_id', 'shop_id'])
fe_data['days_since_historical_low'] = (
    fe_data['date'] - fe_data['historical_low_date']
).dt.days.fillna(0).astype(int)

# Feature 11: Price momentum
print("9. Calculating price momentum...")
fe_data['price_change'] = fe_data.groupby(['app_id', 'shop_id'])['price'].diff()
fe_data['price_pct_change'] = fe_data.groupby(['app_id', 'shop_id'])['price'].pct_change()

# Feature 12: Discount features
print("10. Creating discount features...")
fe_data['discount_amount'] = fe_data['regular_price'] - fe_data['price']
fe_data['is_on_sale'] = (fe_data['sales_percentage'] > 0).astype(int)

# Clean up temporary columns
fe_data = fe_data.drop(columns=['season', 'last_sale_date', 'release_date', 'sale_start', 'sale_end', 'sale_records'], 
                       errors='ignore')

print(f"\n✓ Feature engineering complete!")
print(f"Final dataset shape: {fe_data.shape}")
print(f"\nNew features created:")
new_features = [
    # Temporal
    'days_since_release', 'days_since_last_sale', 'month', 'quarter', 'is_holiday_season',
    'season_Winter', 'season_Spring', 'season_Summer', 'season_Fall',
    # Lag
    'lagged_price_1', 'lagged_price_7', 'price_rolling_mean_7d', 'price_rolling_std_7d',
    'price_rolling_mean_30d', 'price_rolling_std_30d', 'days_since_historical_low',
    # Historical
    'price_min', 'price_max', 'price_mean', 'price_median', 'price_std',
    'price_q25', 'price_q75', 'price_position',
    # Duration
    'sales_duration_days',
    # Discount
    'discount_amount', 'is_on_sale', 
    # Momentum
    'price_change', 'price_pct_change'
]
for feat in new_features:
    print(f"  • {feat}")

=== FEATURE ENGINEERING ===

1. Creating historical price statistics...
2. Calculating sale duration...
3. Tracking last sale date...
4. Adding release date features...
5. Adding seasonal features...
6. Creating lagged price features...
7. Computing rolling statistics...
8. Finding days since historical low...
9. Calculating price momentum...
10. Creating discount features...

✓ Feature engineering complete!
Final dataset shape: (222449, 40)

New features created:
  • days_since_release
  • days_since_last_sale
  • month
  • quarter
  • is_holiday_season
  • season_Winter
  • season_Spring
  • season_Summer
  • season_Fall
  • lagged_price_1
  • lagged_price_7
  • price_rolling_mean_7d
  • price_rolling_std_7d
  • price_rolling_mean_30d
  • price_rolling_std_30d
  • days_since_historical_low
  • price_min
  • price_max
  • price_mean
  • price_median
  • price_std
  • price_q25
  • price_q75
  • price_position
  • sales_duration_days
  • discount_amount
  • is_on_sale
  • price_change


In [5]:
fe_data.fillna(0, inplace=True)

In [6]:
# Save enriched dataset
from pathlib import Path
output_dir = 'data/processed'
# save to data/processed located one level up from the notebooks folder
target_dir = Path.cwd().parent / Path(output_dir)
target_dir.mkdir(parents=True, exist_ok=True)

output_path = target_dir / 'TS_history_engineered.csv'
fe_data.to_csv(output_path, index=False)
print(f"Saved to {output_path}")
# print(f"✓ Engineered dataset saved to 'TS_game_info_engineered.csv'")
# print(f"  Shape: {data.shape}")
# print(f"  Size: {data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Saved to d:\NEU\Time-series Analysis\Steam-Sale-Predict\data\processed\TS_history_engineered.csv


<span style = "font-family: Verdana; font-size: 15px">

### **Feature Engineering Summary**

#### **Historical Price Features**
- **`price_min`**: Minimum historical price per game/shop
- **`price_max`**: Maximum historical price per game/shop
- **`price_mean`**: Average price per game/shop
- **`price_median`**: Median price per game/shop
- **`price_std`**: Price standard deviation (volatility measure)
- **`price_q25`, `price_q75`**: 25th and 75th percentile prices
- **`price_position`**: Normalized price position (0=minimum, 1=maximum)

#### **Sale & Promotion Features**
- **`is_on_sale`**: Binary flag if sales_percentage > 0
- **`discount_amount`**: Absolute discount (regular_price - price)
- **`sales_duration_days`**: Duration of sale period per game/shop combination

#### **Price Momentum & Volatility**
- **`price_change`**: Absolute price change from previous record
- **`price_pct_change`**: Percentage change from previous price
- **`price_rolling_std_7d`**: 7-day rolling standard deviation
- **`price_rolling_std_30d`**: 30-day rolling standard deviation

#### **Temporal & Seasonal Features**
- **`month`**: Month of the year (1-12)
- **`quarter`**: Quarter of the year (1-4)
- **`is_holiday_season`**: Binary flag for Nov-Dec (holiday period)
- **`season_Fall`, `season_Spring`, `season_Summer`, `season_Winter`**: Seasonal one-hot encoding

#### **Lagged & Rolling Averages**
- **`lagged_price_1`**: Previous record price
- **`lagged_price_7`**: Price from 7 records ago
- **`price_rolling_mean_7d`**: 7-day rolling average price
- **`price_rolling_mean_30d`**: 30-day rolling average price

#### **Release & Historical Events**
- **`days_since_release`**: Days elapsed since game release date
- **`days_since_last_sale`**: Days since last recorded sale event
- **`days_since_historical_low`**: Days since the lowest price point was recorded

<span style="font-family: Verdana; font-size: 20px">**Total Features Created: 29**</span>

In [7]:
# Count missing values in fe_data
missing_counts = fe_data.isnull().sum()
missing_pct = (fe_data.isnull().sum() / len(fe_data)) * 100

missing_df = pd.DataFrame({
    'Column': missing_counts.index,
    'Missing_Count': missing_counts.values,
    'Missing_Percentage': missing_pct.values
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

print("Missing Values Summary:\n")
print(missing_df.to_string(index=False))
print(f"\n\nTotal columns with missing values: {len(missing_df)}")

Missing Values Summary:

Empty DataFrame
Columns: [Column, Missing_Count, Missing_Percentage]
Index: []


Total columns with missing values: 0


In [8]:
pd.set_option('display.max_column', None)

In [9]:
fe_data.head()

,app_id,game_title,date,price,regular_price,sales_percentage,shop_id,shop_name,currency,is_historical_low,price_min,price_max,price_mean,price_median,price_std,price_q25,price_q75,price_position,sales_duration_days,days_since_last_sale,days_since_release,month,quarter,season_Fall,season_Spring,season_Summer,season_Winter,is_holiday_season,lagged_price_1,lagged_price_7,price_rolling_mean_7d,price_rolling_std_7d,price_rolling_mean_30d,price_rolling_std_30d,historical_low_date,days_since_historical_low,price_change,price_pct_change,discount_amount,is_on_sale
0,105600,Terraria,2014-10-02,2.49,9.99,75,35,GOG,USD,FALSE,1.99,34.99,7.129241,4.99,4.423811,4.99,9.99,0.015147,4111,-4088.0,0.0,10,4,True,False,False,False,0,2.49,2.49,2.490000,0.000000,2.490000,0.000000,2014-12-04,-63,0.0,0.000000,7.5,1
1,105600,Terraria,2014-10-06,9.99,9.99,0,35,GOG,USD,FALSE,1.99,34.99,7.129241,4.99,4.423811,4.99,9.99,0.242351,4111,-4084.0,0.0,10,4,True,False,False,False,0,2.49,9.99,6.240000,5.303301,6.240000,5.303301,2014-12-04,-59,7.5,3.012048,0.0,0
2,105600,Terraria,2014-11-12,4.99,9.99,50,35,GOG,USD,FALSE,1.99,34.99,7.129241,4.99,4.423811,4.99,9.99,0.090882,4111,-4047.0,0.0,11,4,True,False,False,False,1,9.99,4.99,5.823333,3.818813,5.823333,3.818813,2014-12-04,-22,-5.0,-0.500501,5.0,1
3,105600,Terraria,2014-11-25,9.99,9.99,0,35,GOG,USD,FALSE,1.99,34.99,7.129241,4.99,4.423811,4.99,9.99,0.242351,4111,-4034.0,0.0,11,4,True,False,False,False,1,4.99,9.99,6.865000,3.750000,6.865000,3.750000,2014-12-04,-9,5.0,1.002004,0.0,0
4,105600,Terraria,2014-12-04,1.99,9.99,80,35,GOG,USD,TRUE,1.99,34.99,7.129241,4.99,4.423811,4.99,9.99,0.000000,4111,-4025.0,0.0,12,4,False,False,False,True,1,9.99,1.99,5.890000,3.911521,5.890000,3.911521,2014-12-04,0,-8.0,-0.800801,8.0,1


In [10]:
# Split the dataframe into 2 parts and save them as numbered files
# midpoint = len(data) // 2

# data_part_1 = data.iloc[:midpoint].copy()
# data_part_2 = data.iloc[midpoint:].copy()

# data_part_1.to_csv('history_engineered_part_1.csv', index=False)
# data_part_2.to_csv('history_engineered_part_2.csv', index=False)

# print(f"Saved part 1: {data_part_1.shape} -> history_engineered_part_1.csv")
# print(f"Saved part 2: {data_part_2.shape} -> history_engineered_part_2.csv")
# print(f"  Size: {data_part_1.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
# print(f"  Size: {data_part_2.memory_usage(deep=True).sum() / 1024**2:.2f} MB")